# 📊 Modular Data Sanitization & Exploration Engine

**Assignment: Building a Reusable Python Toolkit for Data Cleaning, Exploration, and Visualization**

---

## Overview

This notebook implements a robust Python toolkit with two main classes:
- **`DataInspector`** — Handles data ingestion, cleaning, imputation, normalization, and visualization
- **`PlottingMethods`** — Modular class for generating custom Plotly charts

The full flow is demonstrated using the **Titanic** dataset:
> Upload → Inspect → Impute → Normalize → Visualize Associations

---

## 1. Installation & Imports

In [41]:
# Install required libraries
!pip install plotly scipy scikit-learn --quiet

In [42]:
import pandas as pd
import numpy as np
import io
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML

# Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scipy / Sklearn for stats and scaling
from scipy import stats
from scipy.stats import pearsonr, pointbiserialr, chi2_contingency, f_oneway
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Colab file upload
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅ All imports successful.")

✅ All imports successful.


---

## 2. `DataInspector` Class — Full Implementation

In [43]:
class DataInspector:
    """
    A comprehensive data cleaning, exploration, and visualization toolkit
    designed for use in Google Colab environments.

    Attributes
    ----------
    df : pd.DataFrame
        The active working DataFrame. Can be loaded via upload_data() or
        assigned directly (inspector.df = pd.read_csv(...)).

    NULL_STRINGS : list
        Strings automatically recognised as NaN during CSV loading.
    """

    # ------------------------------------------------------------------ #
    #  Class-level constants                                               #
    # ------------------------------------------------------------------ #
    NULL_STRINGS = ['?', 'n/a', 'N/A', 'NA', 'na', 'NULL', 'null',
                    'None', 'none', 'NaN', 'nan', '', ' ', '--', '-']

    # ================================================================== #
    # 1. DATA INGESTION & SANITISATION                                    #
    # ================================================================== #

    def upload_data(self):
        """
        Interactively upload a CSV file from the local machine (Colab only).
        Automatically handles garbage strings and attempts numeric type
        conversion on all columns.

        After upload, self.df is populated and a brief summary is printed.
        Outside Colab, raises a RuntimeError with instructions.
        """
        if not IN_COLAB:
            raise RuntimeError(
                "upload_data() is only supported in Google Colab. "
                "Use: inspector.df = pd.read_csv('your_file.csv') instead."
            )

        uploaded = files.upload()
        if not uploaded:
            print("⚠️  No file was uploaded.")
            return

        filename = next(iter(uploaded))
        content  = uploaded[filename]

        self.df = pd.read_csv(
            io.BytesIO(content),
            na_values=self.NULL_STRINGS,
            keep_default_na=True
        )
        self._auto_convert_types()
        print(f"✅ Loaded '{filename}' — {self.df.shape[0]:,} rows × {self.df.shape[1]} columns.")

    def _auto_convert_types(self):
        """
        Internal helper: attempt to coerce object columns to numeric.
        A column is converted only if the result is NOT entirely NaN
        (i.e., at least one value can be interpreted as a number).
        """
        if self.df is None:
            return
        for col in self.df.select_dtypes(include='object').columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if converted.notna().any():          # keep if ANY value converted
                self.df[col] = converted

    # ================================================================== #
    # 2. STRUCTURAL ANALYSIS & CLEANING                                   #
    # ================================================================== #

    def get_summary(self):
        """
        Display a full structural summary of the DataFrame:
        - Shape (rows × columns)
        - First 20 rows preview
        - Column type breakdown (numeric vs. categorical)
        - Descriptive statistics for numeric columns
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()

        print("═" * 60)
        print(f"  DATASET SUMMARY")
        print("═" * 60)
        print(f"  Rows    : {self.df.shape[0]:,}")
        print(f"  Columns : {self.df.shape[1]}")
        print(f"  Numeric : {len(num_cols)} — {num_cols}")
        print(f"  Categ.  : {len(cat_cols)} — {cat_cols}")
        print("─" * 60)
        print("\n📋 First 20 rows:")
        display(self.df.head(20))
        print("\n📊 Numeric descriptive statistics:")
        display(self.df.describe())

    def column_details(self):
        """
        Display per-column information: dtype, non-null count, null count,
        null percentage, and number of unique values.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        info = pd.DataFrame({
            'dtype'       : self.df.dtypes,
            'non_null'    : self.df.notna().sum(),
            'null_count'  : self.df.isna().sum(),
            'null_%'      : (self.df.isna().mean() * 100).round(2),
            'unique'      : self.df.nunique()
        })
        print("\n🔍 Column Details:")
        display(info)

    def get_categorical_summary(self):
        """
        For each categorical column, display value counts and percentage
        distribution.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        cat_cols = self.df.select_dtypes(exclude='number').columns
        if len(cat_cols) == 0:
            print("ℹ️  No categorical columns found.")
            return

        for col in cat_cols:
            vc = self.df[col].value_counts(dropna=False)
            pct = (vc / len(self.df) * 100).round(2)
            summary = pd.DataFrame({'count': vc, 'percentage': pct})
            print(f"\n📌 Column: [{col}]")
            display(summary)

    def show_missing_data(self):
        """
        Print a sorted table of columns with missing values and visualise
        the missing-data pattern as a Plotly bar chart.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        missing = self.df.isna().sum()
        missing = missing[missing > 0].sort_values(ascending=False)

        if missing.empty:
            print("✅ No missing values found!")
            return

        pct = (missing / len(self.df) * 100).round(2)
        tbl = pd.DataFrame({'null_count': missing, 'null_%': pct})
        print("\n🕳️  Missing Data Summary:")
        display(tbl)

        fig = px.bar(
            x=missing.index, y=pct.values,
            labels={'x': 'Column', 'y': 'Missing (%)'},
            title='Missing Data by Column (%)',
            color=pct.values, color_continuous_scale='Reds'
        )
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()

    def handle_missing_values(self, strategy='mean', fill_value=None):
        """
        Impute missing values across the DataFrame.

        Parameters
        ----------
        strategy : str
            One of 'mean', 'median', 'mode', or 'constant'.
        fill_value : scalar, optional
            Used only when strategy='constant'. Must not be None in that case.

        Raises
        ------
        ValueError
            If an unknown strategy is provided, or fill_value is missing
            for the 'constant' strategy.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        valid = {'mean', 'median', 'mode', 'constant'}
        if strategy not in valid:
            raise ValueError(f"strategy must be one of {valid}; got '{strategy}'.")
        if strategy == 'constant' and fill_value is None:
            raise ValueError("Provide fill_value when strategy='constant'.")

        num_cols = self.df.select_dtypes(include='number').columns
        cat_cols = self.df.select_dtypes(exclude='number').columns

        if strategy == 'mean':
            self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean())
            self.df[cat_cols] = self.df[cat_cols].fillna(self.df[cat_cols].mode().iloc[0])
        elif strategy == 'median':
            self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].median())
            self.df[cat_cols] = self.df[cat_cols].fillna(self.df[cat_cols].mode().iloc[0])
        elif strategy == 'mode':
            for col in self.df.columns:
                if self.df[col].isna().any():
                    self.df[col] = self.df[col].fillna(self.df[col].mode().iloc[0])
        elif strategy == 'constant':
            self.df = self.df.fillna(fill_value)

        print(f"✅ Missing values imputed using strategy='{strategy}'.")

    def remove_duplicates(self):
        """
        Remove exact duplicate rows from the DataFrame and report how
        many were dropped.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        before = len(self.df)
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        dropped = before - len(self.df)
        print(f"✅ Removed {dropped:,} duplicate rows. Remaining: {len(self.df):,}.")

    def handle_outliers(self, columns=None, find_and_delete=False):
        """
        Detect outliers using the IQR method on numeric columns.

        Parameters
        ----------
        columns : list of str, optional
            Specific columns to check. Defaults to all numeric columns.
        find_and_delete : bool
            If True, rows containing outliers are removed.
            If False (default), outlier counts are reported without deletion.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        if columns:
            num_cols = [c for c in columns if c in num_cols]

        if not num_cols:
            print("ℹ️  No numeric columns to evaluate.")
            return

        outlier_mask = pd.Series([False] * len(self.df), index=self.df.index)

        report = []
        for col in num_cols:
            Q1  = self.df[col].quantile(0.25)
            Q3  = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            col_mask = (self.df[col] < lower) | (self.df[col] > upper)
            outlier_mask = outlier_mask | col_mask
            report.append({'column': col, 'outliers': col_mask.sum(),
                           'lower_bound': round(lower, 4),
                           'upper_bound': round(upper, 4)})

        print("\n📊 Outlier Report (IQR method):")
        display(pd.DataFrame(report))

        if find_and_delete:
            before = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"✅ Deleted {before - len(self.df):,} outlier rows. Remaining: {len(self.df):,}.")

    def delete_rows(self, indices=None):
        """
        Delete rows by index. If indices is None, prompts the user for
        comma-separated integer indices.

        Parameters
        ----------
        indices : list of int, optional
            Row indices to drop. If None, uses interactive input().
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        if indices is None:
            raw = input("Enter row indices to delete (comma-separated, e.g. 0,3,7): ")
            indices = [int(i.strip()) for i in raw.split(',') if i.strip().isdigit()]

        valid = [i for i in indices if i in self.df.index]
        self.df = self.df.drop(index=valid).reset_index(drop=True)
        print(f"✅ Deleted {len(valid)} row(s). Remaining: {len(self.df):,}.")

    def delete_columns(self, columns=None):
        """
        Delete columns by name. If columns is None, prompts for input.

        Parameters
        ----------
        columns : list of str, optional
            Column names to drop. If None, uses interactive input().
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        if columns is None:
            raw = input("Enter column names to delete (comma-separated): ")
            columns = [c.strip() for c in raw.split(',')]

        valid = [c for c in columns if c in self.df.columns]
        self.df = self.df.drop(columns=valid)
        print(f"✅ Deleted column(s): {valid}. Remaining columns: {list(self.df.columns)}.")

    # ================================================================== #
    # 3. FEATURE ENGINEERING PREPARATION — NORMALISATION                  #
    # ================================================================== #

    def extract_normalized_numeric_data(self, method='minmax'):
        """
        Scale numeric columns and return a new DataFrame of scaled values.

        Parameters
        ----------
        method : str
            'minmax'   — scales to [0, 1]
            'standard' — Z-score normalisation (mean=0, std=1)
            'robust'   — IQR-based scaling (better for outliers)

        Returns
        -------
        pd.DataFrame
            Scaled numeric DataFrame (original DataFrame unchanged).
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return None

        scalers = {
            'minmax'   : MinMaxScaler(),
            'standard' : StandardScaler(),
            'robust'   : RobustScaler()
        }
        if method not in scalers:
            raise ValueError(f"method must be one of {list(scalers.keys())}.")

        num_df = self.df.select_dtypes(include='number').copy()
        if num_df.empty:
            print("ℹ️  No numeric columns to scale.")
            return None

        scaler = scalers[method]
        scaled = pd.DataFrame(
            scaler.fit_transform(num_df.fillna(num_df.median())),
            columns=num_df.columns
        )
        print(f"✅ Numeric data scaled using method='{method}'.")
        return scaled

    def extract_normalized_categorical_data(self, method='onehot'):
        """
        Encode categorical columns and return a new DataFrame.

        Parameters
        ----------
        method : str
            'onehot'  — One-Hot encoding (returns binary indicator columns)
            'ordinal' — Ordinal integer encoding (0, 1, 2, ...)
            'uniform' — Ordinal encoding scaled to [0, 1]

        Returns
        -------
        pd.DataFrame
            Encoded categorical DataFrame (original DataFrame unchanged).
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return None

        cat_df = self.df.select_dtypes(exclude='number').copy()
        if cat_df.empty:
            print("ℹ️  No categorical columns to encode.")
            return None

        # Fill NaN with 'Unknown' placeholder before encoding
        cat_df = cat_df.fillna('Unknown')

        if method == 'onehot':
            enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = enc.fit_transform(cat_df)
            cols    = enc.get_feature_names_out(cat_df.columns)
            result  = pd.DataFrame(encoded, columns=cols)

        elif method in ('ordinal', 'uniform'):
            enc    = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded = enc.fit_transform(cat_df)
            result  = pd.DataFrame(encoded, columns=cat_df.columns)
            if method == 'uniform':
                result = result.apply(
                    lambda s: (s - s.min()) / (s.max() - s.min())
                    if s.max() != s.min() else s
                )
        else:
            raise ValueError("method must be 'onehot', 'ordinal', or 'uniform'.")

        print(f"✅ Categorical data encoded using method='{method}'.")
        return result

    def create_normalized_data_df(self, numeric_method='minmax', cat_method='onehot'):
        """
        Create a single, ML-ready DataFrame by merging scaled numeric columns
        with encoded categorical columns.

        Parameters
        ----------
        numeric_method : str  — see extract_normalized_numeric_data()
        cat_method     : str  — see extract_normalized_categorical_data()

        Returns
        -------
        pd.DataFrame
            Merged, normalised DataFrame ready for machine-learning.
        """
        num_df = self.extract_normalized_numeric_data(method=numeric_method)
        cat_df = self.extract_normalized_categorical_data(method=cat_method)

        parts = [p for p in [num_df, cat_df] if p is not None]
        if not parts:
            print("⚠️  Nothing to merge.")
            return None

        merged = pd.concat(parts, axis=1)
        print(f"✅ Merged DataFrame shape: {merged.shape}")
        display(merged.head())
        return merged

    # ================================================================== #
    # 4. INTERACTIVE VISUALISATIONS                                       #
    # ================================================================== #

    def plot_numerical(self, column_names=None):
        """
        For each specified (or all) numeric column, generate a 3-panel
        subplot:
          1. Horizontal Violin / Box plot
          2. Scatter plot  (index vs. value)
          3. Histogram with KDE overlay

        Parameters
        ----------
        column_names : list of str, optional
            Columns to visualise. Defaults to all numeric columns.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        if column_names:
            num_cols = [c for c in column_names if c in num_cols]

        if not num_cols:
            print("ℹ️  No valid numeric columns found.")
            return

        for col in num_cols:
            series = self.df[col].dropna()
            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=(
                    f'{col} — Violin/Box',
                    f'{col} — Scatter',
                    f'{col} — Histogram'
                )
            )

            # Violin
            fig.add_trace(
                go.Violin(y=series, name=col, box_visible=True,
                          meanline_visible=True, orientation='v',
                          fillcolor='lightblue', line_color='steelblue'),
                row=1, col=1
            )

            # Scatter
            fig.add_trace(
                go.Scatter(x=series.index, y=series, mode='markers',
                           marker=dict(size=4, color='steelblue', opacity=0.6),
                           name=col),
                row=1, col=2
            )

            # Histogram
            fig.add_trace(
                go.Histogram(x=series, name=col, marker_color='steelblue',
                             nbinsx=30, opacity=0.75),
                row=1, col=3
            )

            fig.update_layout(
                title_text=f'Univariate Analysis — {col}',
                height=400, showlegend=False,
                template='plotly_white'
            )
            fig.show()

    def plot_categorical(self, column_names=None):
        """
        For each specified (or all) categorical column, plot a bar chart
        showing raw value counts and percentage labels.

        Parameters
        ----------
        column_names : list of str, optional
            Categorical columns to visualise.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        if column_names:
            cat_cols = [c for c in column_names if c in cat_cols]

        for col in cat_cols:
            vc  = self.df[col].value_counts(dropna=False)
            pct = (vc / len(self.df) * 100).round(1)
            labels = [f"{v} ({p}%)" for v, p in zip(vc.values, pct.values)]

            fig = go.Figure(go.Bar(
                x=vc.index.astype(str), y=vc.values,
                text=labels, textposition='outside',
                marker_color='steelblue'
            ))
            fig.update_layout(
                title=f'Frequency Distribution — {col}',
                xaxis_title=col, yaxis_title='Count',
                xaxis_tickangle=-45,
                template='plotly_white', height=450
            )
            fig.show()

    def plot_relationship(self, col1, col2):
        """
        Intelligently plot the relationship between two columns.

        Chart selection logic:
          - Num–Num   → Scatter with OLS trendline
          - Cat–Num   → Box plot with all data points (strip overlay)
          - Cat–Cat   → Grouped bar chart of cross-frequencies

        Parameters
        ----------
        col1 : str  — First column name.
        col2 : str  — Second column name.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        if col1 not in self.df.columns or col2 not in self.df.columns:
            print(f"⚠️  Column(s) not found: '{col1}', '{col2}'.")
            return

        is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
        is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])

        tmp = self.df[[col1, col2]].dropna()

        if is_num1 and is_num2:
            fig = px.scatter(
                tmp, x=col1, y=col2, trendline='ols',
                title=f'Scatter: {col1} vs {col2}',
                opacity=0.6, template='plotly_white'
            )

        elif (not is_num1 and is_num2) or (is_num1 and not is_num2):
            cat, num = (col1, col2) if not is_num1 else (col2, col1)
            fig = px.box(
                tmp, x=cat, y=num, points='all',
                title=f'Box: {num} by {cat}',
                template='plotly_white'
            )

        else:  # Cat–Cat
            cross = pd.crosstab(tmp[col1], tmp[col2])
            fig = px.bar(
                cross.reset_index().melt(id_vars=col1),
                x=col1, y='value', color='variable', barmode='group',
                title=f'Grouped Bar: {col1} vs {col2}',
                labels={'value': 'Count'},
                template='plotly_white'
            )

        fig.show()

    # ================================================================== #
    # 5. DEEP STATISTICAL INSIGHTS                                        #
    # ================================================================== #

    def plot_numerical_correlation(self):
        """
        Plot a Pearson correlation heatmap for all numeric columns.
        Values close to +1 / −1 indicate strong positive / negative
        linear relationships.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        num_df = self.df.select_dtypes(include='number')
        if num_df.shape[1] < 2:
            print("ℹ️  Need at least 2 numeric columns.")
            return

        corr = num_df.corr(method='pearson')

        fig = px.imshow(
            corr, text_auto='.2f', aspect='auto',
            color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
            title="Pearson Correlation Heatmap (Numeric Columns)"
        )
        fig.update_layout(template='plotly_white')
        fig.show()

    def plot_categorical_correlation(self):
        """
        Plot a Cramér's V association heatmap for all categorical columns.
        Values range 0 (no association) to 1 (perfect association).
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        if len(cat_cols) < 2:
            print("ℹ️  Need at least 2 categorical columns.")
            return

        def cramers_v(x, y):
            ct   = pd.crosstab(x, y)
            chi2 = chi2_contingency(ct, correction=False)[0]
            n    = ct.sum().sum()
            k    = min(ct.shape) - 1
            return np.sqrt(chi2 / (n * k)) if n * k > 0 else 0.0

        tmp  = self.df[cat_cols].fillna('Unknown')
        mat  = pd.DataFrame(np.zeros((len(cat_cols), len(cat_cols))),
                            index=cat_cols, columns=cat_cols)
        for a in cat_cols:
            for b in cat_cols:
                mat.loc[a, b] = 1.0 if a == b else cramers_v(tmp[a], tmp[b])

        fig = px.imshow(
            mat, text_auto='.2f', aspect='auto',
            color_continuous_scale='Blues', zmin=0, zmax=1,
            title="Cramér's V Heatmap (Categorical Columns)"
        )
        fig.update_layout(template='plotly_white')
        fig.show()

    def plot_all_associations_heatmap(self):
        """
        Unified Association Heatmap combining all column types:

        - Num  ↔ Num  : Pearson's r
        - Cat  ↔ Cat  : Cramér's V
        - Num  ↔ Cat  : Point-Biserial (binary cat) or Eta² via ANOVA (multi-cat)

        All values are normalised to [0, 1] so the heatmap is comparable.
        """
        if self.df is None or self.df.empty:
            print("⚠️  No data loaded.")
            return

        cols     = self.df.columns.tolist()
        n        = len(cols)
        assoc    = np.zeros((n, n))

        def _is_num(col):
            return pd.api.types.is_numeric_dtype(self.df[col])

        def _cramers(x, y):
            ct   = pd.crosstab(x, y)
            chi2 = chi2_contingency(ct, correction=False)[0]
            n_   = ct.sum().sum()
            k    = min(ct.shape) - 1
            return np.sqrt(chi2 / (n_ * k)) if n_ * k > 0 else 0.0

        def _eta_squared(num_series, cat_series):
            groups = [num_series[cat_series == g].dropna()
                      for g in cat_series.unique()]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) < 2:
                return 0.0
            F, p = f_oneway(*groups)
            # Convert F to partial eta²
            k   = len(groups) - 1
            N   = sum(len(g) for g in groups)
            SS_between = F * k * (np.var(
                [v for g in groups for v in g]) - np.mean(
                [np.var(g) for g in groups]))
            try:
                eta2 = abs(SS_between / (SS_between + (N - k - 1)))
            except ZeroDivisionError:
                eta2 = 0.0
            return min(1.0, abs(eta2))  # clip to [0,1]

        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    assoc[i, j] = 1.0
                    continue
                if i > j:  # symmetric — reuse
                    assoc[i, j] = assoc[j, i]
                    continue

                pair = self.df[[ci, cj]].dropna()
                if len(pair) < 5:
                    assoc[i, j] = 0.0
                    continue

                ni, nj = _is_num(ci), _is_num(cj)

                if ni and nj:
                    r, _ = pearsonr(pair[ci], pair[cj])
                    assoc[i, j] = abs(r)

                elif not ni and not nj:
                    assoc[i, j] = _cramers(
                        pair[ci].astype(str), pair[cj].astype(str)
                    )

                else:
                    num_col = ci if ni else cj
                    cat_col = cj if ni else ci
                    cat_s   = pair[cat_col].astype(str)

                    if cat_s.nunique() == 2:
                        cat_bin = (cat_s == cat_s.unique()[0]).astype(int)
                        r, _   = pointbiserialr(cat_bin, pair[num_col])
                        assoc[i, j] = abs(r)
                    else:
                        assoc[i, j] = _eta_squared(pair[num_col], cat_s)

        mat = pd.DataFrame(assoc, index=cols, columns=cols)

        fig = px.imshow(
            mat, text_auto='.2f', aspect='auto',
            color_continuous_scale='Viridis', zmin=0, zmax=1,
            title="Unified Association Heatmap (All Column Types)"
        )
        fig.update_layout(template='plotly_white', height=700)
        fig.show()

    # ================================================================== #
    # UTILITY                                                             #
    # ================================================================== #

    def display_image(self, result):
        """
        Render an HTML Plotly figure returned by PlottingMethods.

        Parameters
        ----------
        result : dict
            Dict with key 'html' containing the Plotly HTML string.
        """
        if result and 'html' in result:
            display(HTML(result['html']))
        else:
            print(f"⚠️  Nothing to display. Status: {result.get('status','unknown')}")

print("✅ DataInspector class defined.")

✅ DataInspector class defined.


---

## 3. `PlottingMethods` Class — Full Implementation

In [44]:
class PlottingMethods:
    """
    A modular chart-generation class.

    Every public method accepts either:
      - a ``data`` keyword argument (pd.DataFrame or JSON-serialisable dict/list), or
      - column-name keywords that reference self.df if data is omitted.

    Each method returns a dict::

        {
            'status' : 'success' | 'error',
            'html'   : '<div>...</div>',   # Plotly HTML; present on success
            'message': str                 # error message; present on error
        }

    Use ``display_image(result)`` (or DataInspector.display_image) to render.
    """

    # ------------------------------------------------------------------ #
    #  Helpers                                                            #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _to_df(data):
        """Convert data to pd.DataFrame regardless of input type."""
        if data is None:
            return None
        if isinstance(data, pd.DataFrame):
            return data
        if isinstance(data, (list, dict)):
            return pd.DataFrame(data)
        return None

    @staticmethod
    def _wrap(fig):
        """Convert a Plotly figure to a success result dict."""
        return {
            'status': 'success',
            'html'  : fig.to_html(full_html=False, include_plotlyjs='cdn')
        }

    @staticmethod
    def _err(msg):
        """Return a standardised error result dict."""
        return {'status': 'error', 'message': msg}

    def display_image(self, result):
        """
        Render the HTML contained in a result dict in the notebook.

        Parameters
        ----------
        result : dict  — returned by any plotting method.
        """
        if result and result.get('status') == 'success':
            display(HTML(result['html']))
        else:
            print(f"⚠️  Cannot display. {result.get('message', '')}")

    def get_methods_info(self):
        """
        Return metadata about all available plotting methods.

        Returns
        -------
        dict with key 'response' → list of dicts (method, description, key_params)
        """
        info = [
            {'method': 'plot_bar_chart',
             'description': 'Grouped or stacked bar chart.',
             'key_params': 'x, y, color, barmode, data'},
            {'method': 'plot_pie_chart',
             'description': 'Responsive pie / donut chart.',
             'key_params': 'names, values, hole, title, data'},
            {'method': 'plot_histogram',
             'description': 'Distribution histogram with optional custom bins.',
             'key_params': 'x, bins, title, data'},
            {'method': 'plot_scatter',
             'description': 'Scatter plot with optional colour and size encoding.',
             'key_params': 'x, y, color, size, title, data'},
            {'method': 'plot_heat_map',
             'description': 'Aggregated heatmap (pivot table).',
             'key_params': 'values, index, columns, aggregate_method, title, data'},
            {'method': 'plot_sankey_diagram',
             'description': 'Sankey (flow) diagram between two categorical columns.',
             'key_params': 'source_column, target_column, values, data'},
            {'method': 'plot_simple_sunburst_graph',
             'description': 'Sunburst chart for hierarchical data.',
             'key_params': 'path, values, title, data'},
        ]
        return {'response': info}

    # ------------------------------------------------------------------ #
    #  Chart methods                                                      #
    # ------------------------------------------------------------------ #

    def plot_bar_chart(self, x, y, data=None, color=None,
                       barmode='group', title=None):
        """
        Generate a grouped or stacked bar chart.

        Parameters
        ----------
        x        : str  — column for the x-axis categories.
        y        : str  — column for bar heights.
        data     : pd.DataFrame or JSON-serialisable, optional.
        color    : str, optional — column used to split bars into groups.
        barmode  : str  — 'group' (default) or 'stack'.
        title    : str, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")
        if x not in df.columns or y not in df.columns:
            return self._err(f"Column(s) '{x}' or '{y}' not found in data.")

        fig = px.bar(
            df, x=x, y=y, color=color, barmode=barmode,
            title=title or f'{y} by {x}',
            template='plotly_white'
        )
        return self._wrap(fig)

    def plot_pie_chart(self, names, values, data=None,
                       hole=0.0, title=None):
        """
        Generate a responsive pie (or donut) chart.

        Parameters
        ----------
        names    : str  — column whose unique values become pie slices.
        values   : str  — numeric column for slice sizes.
        data     : pd.DataFrame or JSON-serialisable, optional.
        hole     : float — donut hole size, 0 (solid) to < 1.
        title    : str, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")
        if names not in df.columns or values not in df.columns:
            return self._err(f"Column(s) '{names}' or '{values}' not found.")

        agg = df.groupby(names)[values].sum().reset_index()
        fig = px.pie(
            agg, names=names, values=values, hole=hole,
            title=title or f'{values} Distribution by {names}',
            template='plotly_white'
        )
        return self._wrap(fig)

    def plot_histogram(self, x, data=None, bins=None, title=None):
        """
        Plot a distribution histogram with optional custom bin edges.

        Parameters
        ----------
        x     : str   — numeric column to plot.
        data  : pd.DataFrame or JSON-serialisable, optional.
        bins  : list, optional — custom bin edges, e.g. [0, 18, 35, 60, 100].
        title : str, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")
        if x not in df.columns:
            return self._err(f"Column '{x}' not found.")

        if bins:
            fig = go.Figure(go.Histogram(
                x=df[x].dropna(),
                xbins=dict(start=bins[0], end=bins[-1],
                           size=(bins[-1] - bins[0]) / (len(bins) - 1)),
                marker_color='steelblue', name=x
            ))
            fig.update_layout(
                title=title or f'Histogram — {x}',
                xaxis_title=x, yaxis_title='Count',
                template='plotly_white'
            )
        else:
            fig = px.histogram(
                df, x=x, nbins=30, title=title or f'Histogram — {x}',
                template='plotly_white'
            )
        return self._wrap(fig)

    def plot_scatter(self, x, y, data=None,
                     color=None, size=None, title=None):
        """
        Generate a scatter plot with optional colour and size encoding.

        Parameters
        ----------
        x, y  : str   — column names for axes.
        data  : pd.DataFrame or JSON, optional.
        color : str, optional — column for colour encoding.
        size  : str, optional — numeric column for marker size.
        title : str, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")

        fig = px.scatter(
            df, x=x, y=y, color=color, size=size,
            title=title or f'{y} vs {x}',
            template='plotly_white', opacity=0.7
        )
        return self._wrap(fig)

    def plot_heat_map(self, values, index, columns,
                      aggregade_method='mean', title=None, data=None):
        """
        Build an aggregated pivot-table heatmap.

        Parameters
        ----------
        values           : str  — numeric column to aggregate.
        index            : str  — categorical column for rows.
        columns          : str  — categorical column for columns.
        aggregade_method : str  — 'mean', 'sum', 'count', 'min', or 'max'.
        title            : str, optional.
        data             : pd.DataFrame or JSON, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")

        pivot = df.pivot_table(
            values=values, index=index,
            columns=columns, aggfunc=aggregade_method
        )
        fig = px.imshow(
            pivot, text_auto='.2f', aspect='auto',
            color_continuous_scale='Viridis',
            title=title or f'{values} by {index} & {columns}'
        )
        fig.update_layout(template='plotly_white')
        return self._wrap(fig)

    def plot_sankey_diagram(self, source_column, target_column,
                             values, data=None, title=None):
        """
        Generate a Sankey (flow) diagram between two categorical columns.

        Parameters
        ----------
        source_column : str  — column for flow sources.
        target_column : str  — column for flow targets.
        values        : str  — numeric column aggregated as flow magnitude.
        data          : pd.DataFrame or JSON, optional.
        title         : str, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")

        agg = df.groupby([source_column, target_column])[values] \
                .sum().reset_index()

        all_nodes  = pd.unique(agg[[source_column, target_column]].values.ravel())
        node_index = {n: i for i, n in enumerate(all_nodes)}

        fig = go.Figure(go.Sankey(
            node=dict(label=list(all_nodes), pad=15, thickness=20),
            link=dict(
                source=[node_index[s] for s in agg[source_column]],
                target=[node_index[t] for t in agg[target_column]],
                value =agg[values].tolist()
            )
        ))
        fig.update_layout(
            title=title or f'Sankey: {source_column} → {target_column}',
            template='plotly_white'
        )
        return self._wrap(fig)

    def plot_simple_sunburst_graph(self, path, values, data=None, title=None):
        """
        Generate a Sunburst chart for hierarchical categorical data.

        Parameters
        ----------
        path   : list of str  — ordered list of categorical columns (hierarchy).
        values : str          — numeric column for segment sizes.
        data   : pd.DataFrame or JSON, optional.
        title  : str, optional.

        Returns
        -------
        dict — see class docstring.
        """
        df = self._to_df(data)
        if df is None or df.empty:
            return self._err("No data provided or data is empty.")

        # Validate path columns
        missing = [c for c in path if c not in df.columns]
        if missing:
            return self._err(f"Column(s) not found: {missing}")

        fig = px.sunburst(
            df, path=path, values=values,
            title=title or f'Sunburst: {"/".join(path)}',
            template='plotly_white'
        )
        return self._wrap(fig)

print("✅ PlottingMethods class defined.")

✅ PlottingMethods class defined.


---

## 4. Full Demo — Titanic Dataset

### 4.1 Load Data

In [45]:
# Initialise inspector and load Titanic directly from a public URL
inspector = DataInspector()

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url,
                           na_values=DataInspector.NULL_STRINGS,
                           keep_default_na=True)
inspector._auto_convert_types()
print(f"✅ Loaded Titanic — {inspector.df.shape[0]} rows × {inspector.df.shape[1]} columns.")
inspector.df.head()

✅ Loaded Titanic — 891 rows × 12 columns.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S


### 4.2 Structural Inspection

In [46]:
inspector.get_summary()

════════════════════════════════════════════════════════════
  DATASET SUMMARY
════════════════════════════════════════════════════════════
  Rows    : 891
  Columns : 12
  Numeric : 8 — ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
  Categ.  : 4 — ['Name', 'Sex', 'Cabin', 'Embarked']
────────────────────────────────────────────────────────────

📋 First 20 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C



📊 Numeric descriptive statistics:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,6.610000e+02,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,2.603185e+05,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,4.716093e+05,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,6.930000e+02,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,1.999600e+04,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,2.361710e+05,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,3.477430e+05,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,3.101298e+06,512.329200


In [47]:
inspector.column_details()


🔍 Column Details:


,dtype,non_null,null_count,null_%,unique
PassengerId,int64,891,0,0.00,891
Survived,int64,891,0,0.00,2
Pclass,int64,891,0,0.00,3
Name,object,891,0,0.00,891
Sex,object,891,0,0.00,2
Age,float64,714,177,19.87,88
SibSp,int64,891,0,0.00,7
Parch,int64,891,0,0.00,7
Ticket,float64,661,230,25.81,514
Fare,float64,891,0,0.00,248


In [48]:
inspector.show_missing_data()


🕳️  Missing Data Summary:


,null_count,null_%
Cabin,687,77.10
Ticket,230,25.81
Age,177,19.87
Embarked,2,0.22


### 4.3 Cleaning

In [49]:
# Drop the Cabin column (>77% missing) and PassengerId / Name / Ticket (not predictive)
inspector.delete_columns(columns=['Cabin', 'PassengerId', 'Name', 'Ticket'])

✅ Deleted column(s): ['Cabin', 'PassengerId', 'Name', 'Ticket']. Remaining columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'].


In [50]:
# Impute remaining missing values using median strategy
inspector.handle_missing_values(strategy='median')

✅ Missing values imputed using strategy='median'.


In [51]:
# Remove duplicate rows
inspector.remove_duplicates()

✅ Removed 116 duplicate rows. Remaining: 775.


In [52]:
# Detect and remove outliers in numeric columns
inspector.handle_outliers(columns=['Age', 'Fare', 'SibSp', 'Parch'], find_and_delete=True)


📊 Outlier Report (IQR method):


,column,outliers,lower_bound,upper_bound
0,Age,27,-1.5000,58.5000
1,Fare,102,-31.1718,73.4198
2,SibSp,39,-1.5000,2.5000
3,Parch,15,-1.5000,2.5000


✅ Deleted 174 outlier rows. Remaining: 601.


### 4.4 Categorical Summary & Numeric Distributions

In [53]:
inspector.get_categorical_summary()


📌 Column: [Sex]


,count,percentage
Sex,,
male,398,66.22
female,203,33.78



📌 Column: [Embarked]


,count,percentage
Embarked,,
S,450,74.88
C,102,16.97
Q,49,8.15


In [54]:
# 3-panel univariate plots for numeric columns
inspector.plot_numerical(column_names=['Age', 'Fare', 'SibSp', 'Parch'])

In [55]:
# Categorical frequency charts
inspector.plot_categorical(column_names=['Sex', 'Embarked', 'Pclass'])

### 4.5 Smart Relationship Plots

In [56]:
# Num–Num: Age vs Fare
inspector.plot_relationship('Age', 'Fare')

In [57]:
# Cat–Num: Sex vs Fare
inspector.plot_relationship('Sex', 'Fare')

In [58]:
# Cat–Cat: Sex vs Embarked
inspector.plot_relationship('Sex', 'Embarked')

ValueError: Value of 'color' is not the name of a column in 'data_frame'. Expected one of ['Sex', 'Embarked', 'value'] but received: variable

### 4.6 Feature Engineering — Normalisation

In [61]:
# Scale numeric columns using Robust scaling
scaled_num = inspector.extract_normalized_numeric_data(method='robust')
display(scaled_num.head())

✅ Numeric data scaled using method='robust'.


,Survived,Pclass,Age,SibSp,Parch,Fare
0,0.0,0.0,-0.500000,1.0,0.0,-0.317606
1,1.0,-2.0,0.833333,1.0,0.0,3.219325
2,1.0,0.0,-0.166667,0.0,0.0,-0.280322
3,1.0,-2.0,0.583333,1.0,0.0,2.214956
4,0.0,0.0,0.583333,0.0,0.0,-0.273417


In [60]:
# Encode categorical columns using One-Hot encoding
encoded_cat = inspector.extract_normalized_categorical_data(method='onehot')
display(encoded_cat.head())

✅ Categorical data encoded using method='onehot'.


,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.0,1.0,0.0,0.0,1.0
1,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,1.0


In [59]:
# Create merged ML-ready DataFrame
final_df = inspector.create_normalized_data_df(numeric_method='robust', cat_method='onehot')
final_df.shape

✅ Numeric data scaled using method='robust'.
✅ Categorical data encoded using method='onehot'.
✅ Merged DataFrame shape: (601, 11)


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.0,0.0,-0.500000,1.0,0.0,-0.317606,0.0,1.0,0.0,0.0,1.0
1,1.0,-2.0,0.833333,1.0,0.0,3.219325,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,-0.166667,0.0,0.0,-0.280322,1.0,0.0,0.0,0.0,1.0
3,1.0,-2.0,0.583333,1.0,0.0,2.214956,1.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.583333,0.0,0.0,-0.273417,0.0,1.0,0.0,0.0,1.0


(601, 11)

### 4.7 Association Heatmaps

In [62]:
# Pearson correlation — numeric only
inspector.plot_numerical_correlation()

In [64]:
# Cramér's V — categorical only
inspector.plot_categorical_correlation()

In [63]:
# Unified heatmap — all column types
inspector.plot_all_associations_heatmap()

---

## 5. Custom `PlottingMethods` Demo

In [65]:
PLT = PlottingMethods()

# List all available methods
import pandas as pd
pd.DataFrame(PLT.get_methods_info()['response'])

,method,description,key_params
0,plot_bar_chart,Grouped or stacked bar chart.,"x, y, color, barmode, data"
1,plot_pie_chart,Responsive pie / donut chart.,"names, values, hole, title, data"
2,plot_histogram,Distribution histogram with optional custom bins.,"x, bins, title, data"
3,plot_scatter,Scatter plot with optional colour and size enc...,"x, y, color, size, title, data"
4,plot_heat_map,Aggregated heatmap (pivot table).,"values, index, columns, aggregate_method, titl..."
5,plot_sankey_diagram,Sankey (flow) diagram between two categorical ...,"source_column, target_column, values, data"
6,plot_simple_sunburst_graph,Sunburst chart for hierarchical data.,"path, values, title, data"


In [66]:
# Pie chart: Fare by Passenger Class
result = PLT.plot_pie_chart(names='Pclass', values='Fare',
                            hole=0.35, data=inspector.df,
                            title='Fare Share by Passenger Class')
print(result.get('status'))
PLT.display_image(result)

success


In [ ]:
# Bar chart: Fare by Sex
result = PLT.plot_bar_chart(x='Sex', y='Fare', data=inspector.df,
                            title='Fare by Sex')
PLT.display_image(result)

In [67]:
# Histogram: Age distribution
result = PLT.plot_histogram(x='Age', data=inspector.df,
                            bins=[0, 10, 20, 30, 40, 50, 60, 70, 80],
                            title='Titanic — Age Demographics')
PLT.display_image(result)

In [68]:
# Heatmap: mean Fare by Pclass & Embarked
result = PLT.plot_heat_map(values='Fare', index='Pclass',
                            columns='Embarked',
                            aggregade_method='mean',
                            title='Mean Fare by Class & Embarkation',
                            data=inspector.df)
PLT.display_image(result)

In [ ]:
# Sankey: flow of passengers from Sex → Embarked
result = PLT.plot_sankey_diagram(
    source_column='Sex', target_column='Embarked',
    values='Fare', data=inspector.df,
    title='Passenger Flow: Sex → Embarkation'
)
print(result.get('status'))
PLT.display_image(result)

In [ ]:
# Sunburst: Sex → Embarked hierarchy
result = PLT.plot_simple_sunburst_graph(
    path=['Pclass', 'Sex'], values='Fare',
    data=inspector.df,
    title='Fare Hierarchy: Class → Sex'
)
print(result.get('status'))
PLT.display_image(result)

---

## Summary

| Task | Method |
|------|--------|
| Upload CSV (Colab) | `inspector.upload_data()` |
| Load from URL / path | `inspector.df = pd.read_csv(...)` |
| Structural summary | `inspector.get_summary()` |
| Column details | `inspector.column_details()` |
| Missing data report | `inspector.show_missing_data()` |
| Imputation | `inspector.handle_missing_values(strategy='median')` |
| Remove duplicates | `inspector.remove_duplicates()` |
| IQR outlier removal | `inspector.handle_outliers(find_and_delete=True)` |
| Drop rows / columns | `inspector.delete_rows()` / `inspector.delete_columns()` |
| Scale numerics | `inspector.extract_normalized_numeric_data(method='robust')` |
| Encode categoricals | `inspector.extract_normalized_categorical_data(method='onehot')` |
| ML-ready DataFrame | `inspector.create_normalized_data_df()` |
| Univariate plots | `inspector.plot_numerical([...])` |
| Categorical bar plots | `inspector.plot_categorical([...])` |
| Smart relationship | `inspector.plot_relationship(col1, col2)` |
| Pearson heatmap | `inspector.plot_numerical_correlation()` |
| Cramér's V heatmap | `inspector.plot_categorical_correlation()` |
| Unified heatmap | `inspector.plot_all_associations_heatmap()` |
| Custom charts | `PLT.plot_bar_chart / plot_pie_chart / plot_histogram / ...` |